In [1]:
import pandas as pd
import numpy as np
import gc

BASE = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data'

print("0. LOAD train_clean.parquet VỚI FULL 61 FEATURES...")
train_df = pd.read_parquet(f'{BASE}/train_clean.parquet')

# ── FILTER TỪ 2017-05-01 ĐỂ NHẸ RAM VÀ BẮT TREND NGẮN HẠN ─────────────────────
print("\n1.  Cắt data từ 03/2017...")
train_df = train_df[train_df['date'] >= '2017-03-01'].copy()
train_df['dcoilwtico'] = train_df['dcoilwtico'].astype('float32')
train_df = train_df.sort_values(['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)

# ── TIME FEATURES (Bổ sung nếu data cũ chưa có) ─────────────────────────
print("\n2. Đảm bảo Time features đầy đủ...")
if 'dayofweek' in train_df.columns:
    train_df['dow'] = train_df['dayofweek'].astype('int8')

# ── LAG_140 TỪ TRAIN.CSV GỐC  ─────────────────────────────
print("\n3. Tính lag_140 từ train.csv gốc...")
train_raw = pd.read_csv(
    f'{BASE}/train.csv',
    usecols=['date', 'store_nbr', 'item_nbr', 'unit_sales'],
    dtype={
        'store_nbr' : 'int8',
        'item_nbr'  : 'int32',
        'unit_sales': 'float32'
    },
    parse_dates=['date']
)
train_raw = train_raw[train_raw['date'] >= '2016-12-01'].copy()
train_raw['unit_sales'] = train_raw['unit_sales'].clip(lower=0)
train_raw['unit_sales'] = np.log1p(train_raw['unit_sales']).astype('float32')

sales_map_raw = train_raw.set_index(['store_nbr', 'item_nbr', 'date'])['unit_sales']
del train_raw; gc.collect()

lag_dates = train_df['date'] - pd.Timedelta(days=140)
idx = pd.MultiIndex.from_arrays([
    train_df['store_nbr'].astype('int8'),
    train_df['item_nbr'].astype('int32'),
    lag_dates
])
train_df['lag_140'] = sales_map_raw.reindex(idx).fillna(0).values.astype('float32')
del idx, lag_dates, sales_map_raw; gc.collect()

# ── CATEGORY ──────────────────────────────────────────────
print("\n4. Khóa kiểu dữ liệu Category...")
# Gom toàn bộ nhóm Category ở đây
cat_cols = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class']
for c in cat_cols:
    if c in train_df.columns:
        train_df[c] = train_df[c].astype(str).astype('category')

train_df['perishable'] = train_df['perishable'].astype('int8')

print(f"\n CELL 1 HOÀN TẤT! Shape: {train_df.shape}")

0. LOAD train_clean.parquet VỚI FULL 61 FEATURES...

1.  Cắt data từ 03/2017...

2. Đảm bảo Time features đầy đủ...

3. Tính lag_140 từ train.csv gốc...

4. Khóa kiểu dữ liệu Category...

 CELL 1 HOÀN TẤT! Shape: (27185249, 63)


In [2]:
import pandas as pd
import numpy as np
import gc

print("TẠO MA TRẬN 16 TARGETS...")
train_df['item_nbr'] = train_df['item_nbr'].astype(int)
train_df = train_df.sort_values(['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)

grp_sales = train_df.groupby(['store_nbr', 'item_nbr'], observed=True)['unit_sales']
grp_promo = train_df.groupby(['store_nbr', 'item_nbr'], observed=True)['onpromotion']

print(" Time-shifting 16 ngày...")
for i in range(1, 17):
    train_df[f'target_{i}']       = grp_sales.shift(-i).astype('float32')
    train_df[f'promo_future_{i}'] = grp_promo.shift(-i).fillna(0).astype('int8')

train_df['promo_streak_3'] = (
    train_df['promo_future_1'] +
    train_df['promo_future_2'] +
    train_df['promo_future_3']
)
train_df['promo_streak_7'] = sum(
    train_df[f'promo_future_{i}'] for i in range(1, 8)
).astype('int8')

print(" Target NaN check:")
for i in [1, 8, 16]:
    nan_pct = train_df[f'target_{i}'].isna().mean() * 100
    print(f"   target_{i:2d} NaN: {nan_pct:.1f}%")
print(" Đã tạo xong Ma trận Target!")

TẠO MA TRẬN 16 TARGETS...
 Time-shifting 16 ngày...
 Target NaN check:
   target_ 1 NaN: 0.6%
   target_ 8 NaN: 4.9%
   target_16 NaN: 9.8%
 Đã tạo xong Ma trận Target!


In [3]:
import pandas as pd
import numpy as np
import gc

BASE = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data'

print(" TẠO TẬP TEST BASE (CHỐT TẠI 15/08)...")
X_test_base = train_df[train_df['date'] == '2017-08-15'].copy()

cols_to_drop = [f'target_{i}' for i in range(1, 17)] + ['unit_sales']
X_test_base.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f"   Shape sau filter: {X_test_base.shape}")

print(f"   days_since_last_sale: "
      f"min={X_test_base['days_since_last_sale'].min()}, "
      f"max={X_test_base['days_since_last_sale'].max()}, "
      f"mean={X_test_base['days_since_last_sale'].mean():.1f}")

# Kiểm tra lag_140 trong X_test_base
print(f"   lag_140 mean: {X_test_base['lag_140'].mean():.3f} "
      f"(0 = sai, >0 = đúng)")

print(" Đồng bộ promo_future từ test_clean.parquet...")
test_raw = pd.read_parquet(f'{BASE}/test_clean.parquet')
test_raw['store_nbr'] = test_raw['store_nbr'].astype(str)
test_raw['item_nbr']  = test_raw['item_nbr'].astype(float).fillna(-1).astype(int)

test_promo = test_raw.pivot_table(
    index=['store_nbr', 'item_nbr'],
    columns='date',
    values='onpromotion',
    aggfunc='max'
)
test_promo.columns = [f'promo_future_{i}' for i in range(1, 17)]
test_promo = test_promo.reset_index()

X_test_base.drop(
    columns=[f'promo_future_{i}' for i in range(1, 17)],
    inplace=True, errors='ignore'
)
X_test_base = X_test_base.merge(test_promo, on=['store_nbr', 'item_nbr'], how='left')

for i in range(1, 17):
    X_test_base[f'promo_future_{i}'] = X_test_base[f'promo_future_{i}'].fillna(0).astype('int8')

X_test_base['promo_streak_3'] = (
    X_test_base['promo_future_1'] +
    X_test_base['promo_future_2'] +
    X_test_base['promo_future_3']
)
X_test_base['promo_streak_7'] = sum(
    X_test_base[f'promo_future_{i}'] for i in range(1, 8)
).astype('int8')

before = len(X_test_base)
X_test_base = X_test_base.drop_duplicates(subset=['store_nbr', 'item_nbr'], keep='last')
print(f"   Dedup: {before:,} → {len(X_test_base):,}")

X_test_base['store_nbr'] = X_test_base['store_nbr'].astype(str).astype('category')

del test_promo, test_raw; gc.collect()
print(f" X_test_base sẵn sàng. Shape: {X_test_base.shape}")
print(f"   Duplicate (store,item): {X_test_base.duplicated(['store_nbr','item_nbr']).sum()}")

 TẠO TẬP TEST BASE (CHỐT TẠI 15/08)...
   Shape sau filter: (167515, 80)
   days_since_last_sale: min=1, max=999, mean=13.3
   lag_140 mean: 0.946 (0 = sai, >0 = đúng)
 Đồng bộ promo_future từ test_clean.parquet...
   Dedup: 167,515 → 167,515
 X_test_base sẵn sàng. Shape: (167515, 80)
   Duplicate (store,item): 0


In [4]:
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd
import gc
import json
import os
import shutil
import warnings
warnings.filterwarnings('ignore')

print(" KHỞI ĐỘNG OPTUNA: SĂN THAM SỐ LIGHTGBM ")

# =====================================================================
# 0.  NẠP LẠI LỊCH SỬ OPTUNA TỪ VERSION TRƯỚC (NẾU CÓ)
# =====================================================================
INPUT_DB_PATH = '/kaggle/input/datasets/lminhanhlminhanh/optuna2/optuna_lgbm_v14 (2).db'
INPUT_JSON_PATH = '/kaggle/input/datasets/lminhanhlminhanh/optuna2/optuna_lgbm_params (2).json'

WORKING_DB_PATH = '/kaggle/working/optuna_lgbm_v14.db'
WORKING_JSON_PATH = '/kaggle/working/optuna_lgbm_params.json'

# Phục hồi Database
if os.path.exists(INPUT_DB_PATH) and not os.path.exists(WORKING_DB_PATH):
    shutil.copy(INPUT_DB_PATH, WORKING_DB_PATH)
    print(f" Đã phục hồi thành công Database Optuna từ: {INPUT_DB_PATH}")

# Phục hồi JSON
if os.path.exists(INPUT_JSON_PATH) and not os.path.exists(WORKING_JSON_PATH):
    shutil.copy(INPUT_JSON_PATH, WORKING_JSON_PATH)
    print(f" Đã phục hồi thành công file JSON từ: {INPUT_JSON_PATH}")

# =====================================================================
# 1. DỌN DẸP & CẮT DỮ LIỆU ĐỂ CHỐNG TRÀN RAM
# =====================================================================
if 'train_df' in globals():
    print(" Đang cắt dữ liệu về tháng 03/2017 để bảo vệ Private LB...")
    train_df = train_df[train_df['date'] >= '2017-03-01'].reset_index(drop=True)
    gc.collect()

    print(" Đang dọn dẹp Inf và NaN...")
    numeric_cols = train_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        col_type = train_df[col].dtype
        train_df[col] = train_df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(col_type)

    print(" Đang cắt ranh giới Validation (15/07)...")
    train_mask = train_df['date'] <= '2017-07-15'
    valid_mask = (train_df['date'] > '2017-07-15') & (train_df['date'] <= '2017-07-31')
    
    df_train = train_df[train_mask].copy()
    df_valid = train_df[valid_mask].copy()
    
    cols_to_drop = ['date', 'unit_sales', 'id']
    df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns], inplace=True, errors='ignore')
    df_valid.drop(columns=[c for c in cols_to_drop if c in df_valid.columns], inplace=True, errors='ignore')
    
    del train_df; gc.collect()
    print(" Đã dọn dẹp tập train_df gốc.")
elif 'df_train' not in globals() or 'df_valid' not in globals():
    raise ValueError(" LỖI CRITICAL: Không tìm thấy dữ liệu! Hãy chạy lại các Cell tiền xử lý.")

weight_train = np.where(df_train['perishable'] == 1, 1.25, 1.0).astype(np.float32)
weight_valid = np.where(df_valid['perishable'] == 1, 1.25, 1.0).astype(np.float32)

safe_cat_features = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class']
for col in safe_cat_features:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(str).astype('category')
        df_valid[col] = df_valid[col].astype(str).astype('category')

base_features = [c for c in df_train.columns if c not in safe_cat_features and 'target' not in c and 'promo_future' not in c]

# =====================================================================
# 2.  KHỞI CHẠY VÒNG LẶP TÌM KIẾM
# =====================================================================
if os.path.exists(WORKING_JSON_PATH):
    with open(WORKING_JSON_PATH, 'r') as f:
        best_params_all = json.load(f)
else:
    best_params_all = {}  

representative_days = [2, 7, 14]
TOTAL_TRIALS = 25  # Tổng số vòng muốn hoàn thành cho mỗi mốc ngày

for i in representative_days:
    key_name = f"Ngày_{i}"
    print(f"\n========================================================")
    print(f" ĐANG SĂN THAM SỐ LIGHTGBM CHO NHÓM: {key_name.upper()}/16")
    print(f"========================================================")
    
    features_for_model = base_features + safe_cat_features + [f'promo_future_{i}']
    target_train = df_train[f'target_{i}'].astype('float32')
    target_valid = df_valid[f'target_{i}'].astype('float32')
    
    print(" Đang cắt ngẫu nhiên 30% dữ liệu Train để tăng tốc độ săn mồi...")
    sample_idx = df_train.sample(frac=0.3, random_state=42).index
    weight_train_sample = np.where(df_train.loc[sample_idx, 'perishable'] == 1, 1.25, 1.0).astype(np.float32)
    
    train_data = lgb.Dataset(
        df_train.loc[sample_idx, features_for_model], 
        label=target_train.loc[sample_idx], 
        weight=weight_train_sample, 
        categorical_feature=safe_cat_features,
        free_raw_data=True
    )
    valid_data = lgb.Dataset(
        df_valid[features_for_model], 
        label=target_valid, 
        weight=weight_valid, 
        categorical_feature=safe_cat_features,
        reference=train_data,
        free_raw_data=True
    )
    
    if i <= 3: max_iters = 1500; patience = 50
    elif i <= 10: max_iters = 2000; patience = 75
    else: max_iters = 3000; patience = 100
        
    def objective(trial):
        param = {
            'objective': 'regression_l2',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'n_jobs': -1,
            'seed': 42,
            'verbose': -1,
            'feature_pre_filter': False,
            
            # KHÔNG GIAN TÌM KIẾM ĐÃ CHUẨN HÓA CHO LIGHTGBM
            'num_leaves': trial.suggest_int('num_leaves', 20, 63),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 400),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.9),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.95),
            'bagging_freq': 1,
            'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 10.0),
            'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 10.0),
        }
        
        model = lgb.train(
            param, train_data,
            num_boost_round=max_iters,
            valid_sets=[train_data, valid_data],
            valid_names=['train', 'valid'],
            callbacks=[
                lgb.early_stopping(stopping_rounds=patience, verbose=False)
            ]
        )
        return model.best_score['valid']['rmse']

    # KẾT NỐI DATABASE VÀ ĐỌC LỊCH SỬ CHẠY
    study = optuna.create_study(
        study_name=f"lgbm_study_{key_name}", 
        storage=f"sqlite:///{WORKING_DB_PATH}", 
        direction='minimize', 
        load_if_exists=True
    )
    
    n_trials_done = len([t for t in study.trials if t.state.name == 'COMPLETE'])
    n_trials_to_run = max(0, TOTAL_TRIALS - n_trials_done)

    if n_trials_to_run > 0:
        print(f" Đã hoàn thành {n_trials_done}/{TOTAL_TRIALS} vòng. Bắt đầu chạy {n_trials_to_run} vòng còn thiếu...")
        # Để chạy ngầm an toàn, mỗi lần Save & Run All chỉ nên chạy tầm 10-15 vòng
        # Ở đây min(n_trials_to_run, 15) giúp Kaggle không bị Time Out 9 tiếng
        trials_this_run = min(n_trials_to_run, 15) 
        print(f" Phiên chạy hiện tại sẽ chạy {trials_this_run} vòng để tránh Time Out.")
        study.optimize(objective, n_trials=trials_this_run)
    else:
        print(f" Đã hoàn thành đủ {TOTAL_TRIALS} vòng từ trước. Bỏ qua.")

    best_params_all[key_name] = study.best_params
    print(f"\n Xong {key_name}! RMSE tốt nhất: {study.best_value:.5f}")
    
    with open(WORKING_JSON_PATH, 'w') as f:
        json.dump(best_params_all, f, indent=4)
    
    del train_data, valid_data, target_train, target_valid; gc.collect()

print("\n BỘ THAM SỐ VÀNG LIGHTGBM:")
for k, v in best_params_all.items():
    print(f" {k}: {v}")

 KHỞI ĐỘNG OPTUNA: SĂN THAM SỐ LIGHTGBM 
 Đã phục hồi thành công Database Optuna từ: /kaggle/input/datasets/lminhanhlminhanh/optuna2/optuna_lgbm_v14 (2).db
 Đã phục hồi thành công file JSON từ: /kaggle/input/datasets/lminhanhlminhanh/optuna2/optuna_lgbm_params (2).json
 Đang cắt dữ liệu về tháng 03/2017 để bảo vệ Private LB...
 Đang dọn dẹp Inf và NaN...
 Đang cắt ranh giới Validation (15/07)...
 Đã dọn dẹp tập train_df gốc.

 ĐANG SĂN THAM SỐ LIGHTGBM CHO NHÓM: NGÀY_2/16
 Đang cắt ngẫu nhiên 30% dữ liệu Train để tăng tốc độ săn mồi...


[I 2026-05-02 02:37:02,587] Using an existing study with name 'lgbm_study_Ngày_2' instead of creating a new one.


 Đã hoàn thành đủ 25 vòng từ trước. Bỏ qua.

 Xong Ngày_2! RMSE tốt nhất: 0.57579

 ĐANG SĂN THAM SỐ LIGHTGBM CHO NHÓM: NGÀY_7/16
 Đang cắt ngẫu nhiên 30% dữ liệu Train để tăng tốc độ săn mồi...


[I 2026-05-02 02:37:16,734] Using an existing study with name 'lgbm_study_Ngày_7' instead of creating a new one.


 Đã hoàn thành đủ 25 vòng từ trước. Bỏ qua.

 Xong Ngày_7! RMSE tốt nhất: 0.59821

 ĐANG SĂN THAM SỐ LIGHTGBM CHO NHÓM: NGÀY_14/16
 Đang cắt ngẫu nhiên 30% dữ liệu Train để tăng tốc độ săn mồi...


[I 2026-05-02 02:37:37,393] Using an existing study with name 'lgbm_study_Ngày_14' instead of creating a new one.


 Đã hoàn thành đủ 25 vòng từ trước. Bỏ qua.

 Xong Ngày_14! RMSE tốt nhất: 0.60954

 BỘ THAM SỐ VÀNG LIGHTGBM:
 Ngày_2: {'num_leaves': 52, 'min_data_in_leaf': 394, 'learning_rate': 0.052717010242044234, 'feature_fraction': 0.50475953078065, 'bagging_fraction': 0.8746874024163661, 'lambda_l1': 3.858328658906121, 'lambda_l2': 9.98523705928188}
 Ngày_7: {'num_leaves': 40, 'min_data_in_leaf': 372, 'learning_rate': 0.03795249387056751, 'feature_fraction': 0.5306474002473422, 'bagging_fraction': 0.7136021136297174, 'lambda_l1': 1.5736064003560881, 'lambda_l2': 0.9499313436853944}
 Ngày_14: {'num_leaves': 59, 'min_data_in_leaf': 342, 'learning_rate': 0.03608595127848215, 'feature_fraction': 0.5698630000942243, 'bagging_fraction': 0.8601165719677516, 'lambda_l1': 3.7405472341742834, 'lambda_l2': 9.88849544966942}


In [5]:
import lightgbm as lgb
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import gc
import json
import os
import warnings
warnings.filterwarnings('ignore')

print("🚀 BẮT ĐẦU LIGHTGBM (PHIÊN BẢN CHỐNG TIMEOUT - GIỮ NGUYÊN FEATURE)")

# =====================================================================
# 1. BẢO VỆ RAM & KIỂM TRA DỮ LIỆU
# =====================================================================
if 'train_df' in globals():
    print("📋 Đang tách tập Train và Validation...")
    train_mask = train_df['date'] <= '2017-07-15'
    valid_mask  = (train_df['date'] > '2017-07-15') & (train_df['date'] <= '2017-07-31')

    df_train = train_df[train_mask].copy()
    df_valid  = train_df[valid_mask].copy()
    del train_df; gc.collect()
elif 'df_train' in globals() and 'df_valid' in globals():
    print("✅ Đã nhận bàn giao df_train và df_valid.")
else:
    raise ValueError("❌ LỖI: Không tìm thấy dữ liệu!")

# =====================================================================
# 2. KHAI BÁO CATEGORY & LỌC FEATURE (GIỮ NGUYÊN 61 SIÊU TÍNH NĂNG)
# =====================================================================
safe_cat_features = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class']

exclude_cols = set(safe_cat_features) | \
               {'date', 'unit_sales', 'id', 'perishable'} | \
               {f'target_{i}' for i in range(1, 17)} | \
               {f'promo_future_{i}' for i in range(1, 17)}

base_features = [c for c in df_train.columns if c not in exclude_cols]

keep_cols = (
    base_features + safe_cat_features +
    ['perishable'] +
    [f'target_{i}' for i in range(1, 17)] +
    [f'promo_future_{i}' for i in range(1, 17)]
)
keep_cols = list(dict.fromkeys([c for c in keep_cols if c in df_train.columns]))
df_train = df_train[keep_cols]
df_valid  = df_valid[keep_cols]

weight_train = np.where(df_train['perishable'] == 1, 1.25, 1.0)
weight_valid  = np.where(df_valid['perishable'] == 1, 1.25, 1.0)

for col in safe_cat_features:
    if col in df_train.columns:
        df_train[col]    = df_train[col].astype('category')
        df_valid[col]    = df_valid[col].astype('category')
        if col in X_test_base.columns:
            X_test_base[col] = X_test_base[col].astype('category')

# =====================================================================
# 3. THAM SỐ TỐI ƯU TỐC ĐỘ (BASE_PARAMS)
# =====================================================================
BASE_PARAMS = {
    'objective'       : 'regression_l2',
    'metric'          : 'rmse',
    'boosting_type'   : 'gbdt', 
    'data_sample_strategy': 'goss', #  Tăng tốc độ cực nhanh cho dữ liệu lớn
    'force_row_wise'  : True,
    'max_bin'         : 63,         #  Giảm gánh nặng tính toán histogram
    'verbose'         : -1,
    'seed'            : 42,
    'n_jobs'          : -1, 
}

# Load Optuna params
PARAMS_FILE_LGB = '/kaggle/working/optuna_lgbm_params.json'
loaded_optuna_params = {}
if os.path.exists(PARAMS_FILE_LGB):
    with open(PARAMS_FILE_LGB, 'r') as f:
        loaded_optuna_params = json.load(f)

all_test_predictions = []

# =====================================================================
# 4. HUẤN LUYỆN 16 MODELS (SIẾT CHẶT THỜI GIAN)
# =====================================================================
for i in tqdm(range(1, 17), desc="Train 16 Models"):
    # Cấu hình linh hoạt để tránh Timeout ở các model cuối
    if i <= 3: 
        optuna_key = "Ngày_2"; patience = 30; max_iters = 800
    elif i <= 10: 
        optuna_key = "Ngày_7"; patience = 30; max_iters = 1000
    else: 
        optuna_key = "Ngày_14"; patience = 30; max_iters = 1100 # Giới hạn vòng lặp cho dự báo xa

    lgb_params = BASE_PARAMS.copy()
    if optuna_key in loaded_optuna_params:
        lgb_params.update(loaded_optuna_params[optuna_key])
        # Đảm bảo max_bin luôn là 63 để giữ tốc độ
        lgb_params['max_bin'] = 63

    features_for_model = base_features + safe_cat_features + [f'promo_future_{i}']
    
    # Tạo Dataset với free_raw_data=True để tiết kiệm RAM
    train_data = lgb.Dataset(
        df_train[features_for_model], label=df_train[f'target_{i}'].fillna(0),
        weight=weight_train, categorical_feature=safe_cat_features,
        free_raw_data=True 
    )
    valid_data = lgb.Dataset(
        df_valid[features_for_model], label=df_valid[f'target_{i}'].fillna(0),
        weight=weight_valid, categorical_feature=safe_cat_features,
        reference=train_data, free_raw_data=True
    )

    model = lgb.train(
        lgb_params, train_data,
        num_boost_round=max_iters,
        valid_sets=[valid_data], # Chỉ cần valid_sets này để tính early stopping nhanh hơn
        callbacks=[
            lgb.early_stopping(stopping_rounds=patience),
            lgb.log_evaluation(period=200),
        ],
    )

    # Predict ngay để giải phóng model nhanh
    pred = model.predict(X_test_base[features_for_model], num_iteration=model.best_iteration)
    pred = np.expm1(np.clip(pred, 0, 10))

    pred_df = X_test_base[['store_nbr', 'item_nbr']].copy()
    pred_df['date']       = pd.to_datetime('2017-08-15') + pd.Timedelta(days=i)
    pred_df['unit_sales'] = pred
    all_test_predictions.append(pred_df)

    # Dọn dẹp triệt để sau mỗi vòng lặp
    del train_data, valid_data, model; gc.collect()

print("\n🎉 XONG! 16 mô hình đã hoàn tất .")

🚀 BẮT ĐẦU LIGHTGBM (PHIÊN BẢN CHỐNG TIMEOUT - GIỮ NGUYÊN FEATURE)
✅ Đã nhận bàn giao df_train và df_valid.


Train 16 Models:   0%|          | 0/16 [00:00<?, ?it/s]

Training until validation scores don't improve for 30 rounds
[200]	valid_0's rmse: 0.573583
[400]	valid_0's rmse: 0.571073
[600]	valid_0's rmse: 0.569683
[800]	valid_0's rmse: 0.569109
Did not meet early stopping. Best iteration is:
[794]	valid_0's rmse: 0.569095
Training until validation scores don't improve for 30 rounds
[200]	valid_0's rmse: 0.579512
[400]	valid_0's rmse: 0.577734
[600]	valid_0's rmse: 0.576832
[800]	valid_0's rmse: 0.576307
Did not meet early stopping. Best iteration is:
[796]	valid_0's rmse: 0.576272
Training until validation scores don't improve for 30 rounds
[200]	valid_0's rmse: 0.584398
[400]	valid_0's rmse: 0.582693
[600]	valid_0's rmse: 0.581408
[800]	valid_0's rmse: 0.580874
Did not meet early stopping. Best iteration is:
[799]	valid_0's rmse: 0.580872
Training until validation scores don't improve for 30 rounds
[200]	valid_0's rmse: 0.592336
[400]	valid_0's rmse: 0.59024
[600]	valid_0's rmse: 0.589099
[800]	valid_0's rmse: 0.588355
[1000]	valid_0's rmse: 0

In [6]:
import pandas as pd
import numpy as np

BASE = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data'

print("\n Tạo file nộp bài...")
final_pred_df = pd.concat(all_test_predictions, ignore_index=True)

dup = final_pred_df.duplicated(['store_nbr', 'item_nbr', 'date']).sum()
if dup > 0:
    final_pred_df = final_pred_df.drop_duplicates(
        ['store_nbr', 'item_nbr', 'date'], keep='last'
    )
print(f"final_pred_df shape: {final_pred_df.shape}")

test_raw = pd.read_parquet(f'{BASE}/test_clean.parquet')
test_raw['date']           = pd.to_datetime(test_raw['date'])
final_pred_df['date']      = pd.to_datetime(final_pred_df['date'])
test_raw['store_nbr']      = test_raw['store_nbr'].astype(str)
final_pred_df['store_nbr'] = final_pred_df['store_nbr'].astype(str)
test_raw['item_nbr']       = test_raw['item_nbr'].astype(float).fillna(-1).astype(int)
final_pred_df['item_nbr']  = final_pred_df['item_nbr'].astype(int)

submission = test_raw[['id', 'store_nbr', 'item_nbr', 'date']].merge(
    final_pred_df, on=['store_nbr', 'item_nbr', 'date'], how='left'
)

submission['unit_sales'] = submission['unit_sales'].fillna(0)

print(f"\nSubmission shape: {submission.shape}")
print(f"Duplicate IDs: {submission['id'].duplicated().sum()}")
print(f"NaN: {submission['unit_sales'].isna().sum()}")
print(f"Zero: {(submission['unit_sales']==0).sum():,} ({(submission['unit_sales']==0).mean()*100:.1f}%)")
print(f"Stats: min={submission['unit_sales'].min():.2f}, "
      f"max={submission['unit_sales'].max():.2f}, "
      f"mean={submission['unit_sales'].mean():.2f}")

assert submission['id'].duplicated().sum() == 0, " DUPLICATE ID!"

submission[['id', 'unit_sales']].to_csv(
    '/kaggle/working/submission_lgb.csv', index=False
)
print(" Đã lưu: submission_lgb.csv")


 Tạo file nộp bài...
final_pred_df shape: (2680240, 4)

Submission shape: (3370464, 5)
Duplicate IDs: 0
NaN: 0
Zero: 813,390 (24.1%)
Stats: min=0.00, max=306.76, mean=2.78
 Đã lưu: submission_lgb.csv
